# Lidocaine: 30 Hz burst vs ARC-EX, before and with lidocaine

**The question.** Lidocaine over the skin blocks cutaneous afferents. If a protocol recruits the
cord mainly through **dorsal-root afferents**, blocking the skin should leave it largely alone;
if a protocol drives **skin and superficial fibres** as well, blocking them should cost it more.
So: does lidocaine cost ARC-EX more than it costs 30 Hz, or the same?

**Two participants, one session each.**

| | | |
|---|---|---|
| **P04** | 17-07-2026, electrode 1 | before → lidocaine, both protocols |
| **P05** | 24-07-2026 (`testSCS`), electrode 2, **cathodic** | before → lidocaine, both protocols |

P04's electrode 3 and P05's anodic blocks exist in the session folders and are analysed in the
per-session notebooks; this comparison keeps one montage per participant so the two are read the
same way.

**Two things are measured, and they answer different halves of the question.**

1. **The threshold moves** — how much more current each protocol needs once the skin is blocked.
   This uses the thresholds you picked, nothing else.
2. **The response at a matched intensity shrinks** — with lidocaine as a % of the same
   muscle's response *before*, both read at the same mA, so the comparison is of the response
   and not of the current.

**The rule for intensity.** Every muscle is analysed **at its own motor threshold**, picked by
hand in section 2 — never at one mA for the whole arm, and never a step above. Within a protocol,
before and lidocaine are read at the **same** intensity (the before-threshold), or the bars would
compare current instead of lidocaine. 30 Hz and ARC-EX are never forced onto one intensity: they
sit ~70 mA apart and each carries its own thresholds.

**Colours, as everywhere else:** 30 Hz burst **grey**, ARC-EX **red**; the paler shade is
*before*, the solid one *with lidocaine*.

**Millivolts never travel between participants** — different electrodes, different skin, different
amplifier gain. Every comparison below is within a participant, and only the *changes* are read
across them.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       condition_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       mt_stamp, warn_if_stale,
                       fig_threshold_grid, fig_detection_grid, fig_group_summary, fig_group_panels)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D04 = "tSCS_CHUV_data/17-07-2026/P04tscsHealthy/"    # electrode 1
D06 = "tSCS_CHUV_data/28-08-2026/"                   # P06, 28-09-2026, electrode 2, CATHODIC
D05 = "tSCS_CHUV_data/24-07-2026/testSCS/"           # electrode 2, CATHODIC only:
#   that session also holds the anodic blocks (Burst 102721 / 114055, Modulated 103849 /
#   114730). They are the polarity comparison, in P05_cathodic_vs_anodic.ipynb - mixing
#   one in here would compare lidocaine with polarity.

RUN = {   # (subject, condition) -> csv
  ("P04", "30 Hz · before"):    D04 + "Burst_autosave_20260717_150152_655ms.csv",       # 15:01
  ("P04", "30 Hz · lidocaine"): D04 + "Burst_autosave_20260717_161911_499ms.csv",       # 16:19
  ("P04", "ARC-EX · before"):   D04 + "Modulated_autosave_20260717_150657_684ms.csv",   # 15:06
  ("P04", "ARC-EX · lidocaine"):D04 + "Modulated_autosave_20260717_162347_270ms.csv",   # 16:23
  ("P05", "30 Hz · before"):    D05 + "Burst_autosave_20260724_102443_670ms.csv",       # 10:24
  ("P05", "30 Hz · lidocaine"): D05 + "Burst_autosave_20260724_113834_522ms.csv",       # 11:38
  ("P05", "ARC-EX · before"):   D05 + "Modulated_autosave_20260724_103530_508ms.csv",   # 10:35
  ("P05", "ARC-EX · lidocaine"):D05 + "Modulated_autosave_20260724_114332_473ms.csv",   # 11:43
  ("P06", "30 Hz · before"):    D06 + "Burst_autosave_20260928_152710_593ms.csv",       # 15:27
  ("P06", "30 Hz · lidocaine"): D06 + "Burst_autosave_20260928_165756_941ms.csv",       # 16:58
  ("P06", "ARC-EX · before"):   D06 + "Modulated_autosave_20260928_154023_427ms.csv",   # 15:40
  ("P06", "ARC-EX · lidocaine"):D06 + "Modulated_autosave_20260928_170517_985ms.csv",   # 17:05
}
SUBJECTS   = ["P04", "P05", "P06"]
PROTOCOLS  = ["30 Hz", "ARC-EX"]
PHASES     = ["before", "lidocaine"]
CONDITIONS = [f"{p} · {ph}" for p in PROTOCOLS for ph in PHASES]

# which protocol each condition belongs to: conditions in different groups are never compared
# and never share an intensity
GROUP = {f"{p} · {ph}": p for p in PROTOCOLS for ph in PHASES}
# ...and each one is read as a % of its OWN protocol's "before" block
BASE  = {f"{p} · {ph}": f"{p} · before" for p in PROTOCOLS for ph in PHASES}

# 30 Hz grey, ARC-EX red; pale = before, solid = with lidocaine
COLOUR = {"30 Hz · before": "#C6C6C6", "30 Hz · lidocaine": "#9A9A9A",
          "ARC-EX · before": "#E3A9A1", "ARC-EX · lidocaine": "#C0392B"}
PROTO_COLOUR = {"30 Hz": "#9A9A9A", "ARC-EX": "#C0392B"}

# the same four muscles as the baseline and vibration comparisons
MUSCLES = ["Flex. digitorum (L)", "Flex. digitorum (R)", "Ext. digitorum (L)",
           "Flex. carpi rad. (R)"]

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN
for s in SUBJECTS:
    for cond in CONDITIONS:
        f = runs[(s, cond)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{s}  {cond:20s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA, "
              f"step {amps[1] - amps[0]:g}")

## 2 · The motor thresholds you chose

**Only the six `before` rows are used by the analysis.** Each group — a participant and a protocol
— takes the threshold picked in **its own `before lidocaine` recording**, and that group's before
and lidocaine blocks are both read at that one intensity. 30 Hz and ARC-EX never share a
threshold, and neither do two participants: every group stands on its own control.

That is the only way lidocaine can be measured. If the lidocaine block were read at its own
threshold, a change in the response and a change in the current would be indistinguishable.

The `lidocaine` rows are shown here because the grid is the check on what is saved, and because a
threshold that moves between blocks is itself a result — section 4a is exactly that. They do not
set the intensity anything is measured at.

Read straight from `results/<session>/mt_*.csv`; `not picked` = nothing saved, and the cell below
says whether each group's threshold is yours or the detector's.

In [ ]:
CHECK = {f"{s} · {cond}": runs[(s, cond)] for s in SUBJECTS for cond in CONDITIONS}

picked = fig_threshold_grid(CHECK, MUSCLES,
                            title="the thresholds saved for each recording")

### The numbers, and where each one came from

In [ ]:
# hand-picked thresholds win wherever they exist; detection fills in the rest. Pick them in
# section 5 or in notebooks/motor_thresholds/ and re-run this cell - the source is printed below.
MT = {}      # MT[(subject, protocol)] = {muscle: mA}, from that protocol's BEFORE block
for s in SUBJECTS:
    for p in PROTOCOLS:
        csv = runs[(s, f"{p} · before")]
        MT[(s, p)] = thresholds_for(csv, MUSCLES, consecutive=2, **KW)
        print(f"{s}  {p:7s} before   {threshold_source(csv)}")

MT_ALL = {lab: picked.get(lab, {}) for lab in CHECK}      # every block's own picks

print(f"\nmotor threshold (mA), as picked in each block")
print(f"{'muscle':22s}" + "".join(f"{s + ' ' + c.replace(' · ', ' '):>20s}"
                                  for s in SUBJECTS for c in CONDITIONS))
for m in MUSCLES:
    print(f"{m:22s}" + "".join(f"{str(MT_ALL[f'{s} · {c}'].get(m) or '-'):>20s}"
                               for s in SUBJECTS for c in CONDITIONS))

MT_STAMP = mt_stamp(runs)      # so the figures can tell you if a pick is saved after this

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2 — a row per recording, a column per muscle — but showing how the peaks
are found at the threshold picked in **that** recording. The 10 pulses are re-aligned on their own
onset, with the max (v) and min (^) the peak-to-peak is made of.

| | |
|---|---|
| **green** | the response window — `RESP_START_MS` after the pulse to `GUARD_MS` before the next |
| **red bands** | the only places a peak may be taken: `ANCHOR_WIN_MS` either side of the train's own average latency. **A peak outside these is skipped however clear it looks** |
| markers stacked | the same deflection found on every pulse |
| red ring | flagged — on a window border, or at a different latency from the rest |
| corner | `clean`, or how many of the 10 pulses were flagged |

A lidocaine block whose train sits in the noise is a *result*, not a detection failure — check
here which of the two it is before believing a small bar in section 4.

### If a peak you can see is not being taken

`TRY` changes the settings **for this figure only**, so you can test whether detection is too
strict without touching any result. When you find settings you want, copy them into `KW` in
section 1.

| setting | now | loosen it by | what it controls |
|---|---|---|---|
| `anchor_win_ms` | 3.0 | **raising** it, or `anchor=None` | how far from the train's average latency a peak may sit — much the most common reason a visible response is missed |
| `resp_start_ms` | 8.0 | **lowering** it | when the window opens after the pulse; too low and it takes the stimulus artifact |
| `min_snr` | 1.2 | **lowering** it, or `None` | how far above that muscle's noise a train must be to count at all |
| `max_edge_frac` | 0.5 | **raising** it, or `None` | how many pulses may peak on a window border before the whole train is called artifact |

In [ ]:
# ======== try different detection settings, for this figure only ========
TRY = dict(KW)                     # start from the analysis settings...
# TRY["anchor_win_ms"] = 6.0       # ...and loosen one at a time: widen the red bands
# TRY["anchor"] = None             # no anchoring: biggest max/min anywhere in the window
# TRY["resp_start_ms"] = 6.0       # open the window sooner after the pulse
# TRY["min_snr"] = None            # stop discarding trains that sit close to noise
# TRY["max_edge_frac"] = None      # stop discarding trains that peak on a window border
# ========================================================================

fig_detection_grid(CHECK, MT_ALL, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title="peak detection at each recording's own threshold", **TRY)

## 4 · The results

### 4a · How far lidocaine moved the threshold

The threshold you picked **with lidocaine** as a % of the one you picked **before**, in the same
muscle, same protocol, same electrode. **100 % = the skin block cost that protocol nothing.**
Above 100 % means it needed more current once cutaneous afferents were blocked — and the
comparison that matters is **grey against red in the same muscle**: if ARC-EX rises more than
30 Hz does, more of what ARC-EX was recruiting came through the skin.

The raw milliamps are printed underneath, because a % of a 35 mA threshold and a % of a 105 mA
one are not the same thing in current.

In [ ]:
shift = {}
for s in SUBJECTS:
    for p in PROTOCOLS:
        b = MT_ALL[f"{s} · {p} · before"]
        a = MT_ALL[f"{s} · {p} · lidocaine"]
        for m in MUSCLES:
            if b.get(m) and a.get(m):
                shift[(s, p, m)] = dict(pct=100 * a[m] / b[m], before=b[m], after=a[m])

fig_vibration(shift, SUBJECTS, MUSCLES, PROTOCOLS, colours=PROTO_COLOUR,
              min_snr_ratio=2.0,   # a bar under 2x its noise is left out and
                                   # the axis says whether it was weak or absent
              ylabel="% of before", ref=100, ylim=(0, 150),
              titles={s: s for s in SUBJECTS},
              title="Motor threshold with lidocaine, % of before")

print(f"{'subject':8s}{'muscle':22s}" + "".join(f"{p + ' before':>16s}{p + ' lido':>14s}{'%':>7s}"
                                                for p in PROTOCOLS))
for s in SUBJECTS:
    for m in MUSCLES:
        row = ""
        for p in PROTOCOLS:
            v = shift.get((s, p, m))
            row += (f"{v['before']:>14g} mA{v['after']:>12g} mA{v['pct']:>6.0f} %" if v
                    else f"{'-':>16s}{'-':>14s}{'-':>7s}")
        print(f"{s:8s}{m:22s}{row}")

### 4b · The response at a matched intensity

Each protocol read at the intensity its **before** block defines, so the only thing that changes
between the two bars is the lidocaine. **100 % = the response did not change.** Grey is 30 Hz, red
is ARC-EX; the pale bars are the *before* blocks and sit at 100 % by construction — they are there
so the pair can be seen as a pair.

Read it against 4a: a protocol can lose response *and* need more current, or only one of the two.

In [ ]:
warn_if_stale(runs, MT_STAMP, used=[(s, f"{p} · before") for s in SUBJECTS for p in PROTOCOLS])

tab = condition_table(runs, MT, MUSCLES, CONDITIONS, base=BASE, match="common",
                      group=GROUP, **KW)
fig_vibration(tab, SUBJECTS, MUSCLES, CONDITIONS, colours=COLOUR,
              min_snr_ratio=2.0,   # a bar under 2x its noise is left out and
                                   # the axis says whether it was weak or absent
              ylabel="% of before", ylim=(0, 200),   # bars past the top are marked ^ with
                                                     # their real value printed above them
              titles={s: s for s in SUBJECTS},
              title="Response with lidocaine, % of before at the same intensity")

### The numbers, so they can be checked

In [ ]:
print(f"{'subject':8s}{'muscle':22s}" + "".join(f"{p:>12s}{'mA':>7s}" for p in PROTOCOLS))
for s in SUBJECTS:
    for m in MUSCLES:
        row = ""
        for p in PROTOCOLS:
            v = tab.get((s, f"{p} · lidocaine", m), {})
            pct, amp = v.get("pct", float("nan")), v.get("amp")
            row += f"{(f'{pct:.0f} %' if pct == pct else '-'):>12s}{(f'{amp:g}' if amp else '-'):>7s}"
        print(f"{s:8s}{m:22s}{row}")

### 4c · One figure per muscle — before and with lidocaine, with the EMG

The 1st pulse in mV, the 2nd as a % of it, and the mean of pulses 2–10 as a % of it, with the EMG
underneath in matching colours. The mV panel is the quantity the two percentages are a percentage
**of**: read it within a participant only.

Along the train is a second, independent readout — if lidocaine removes a late, polysynaptic
component, the 2nd pulse and the 2–10 mean fall even where the 1st pulse barely moves.

In [ ]:
# the intensities the table above settled on, so these figures and that one cannot disagree
MT_cond = {(s, cond): {m: tab[(s, cond, m)]["amp"] for m in MUSCLES if (s, cond, m) in tab}
           for s in SUBJECTS for cond in CONDITIONS}

for m in MUSCLES:
    fig_bars(runs, MT_cond, m, SUBJECTS, protocols=tuple(CONDITIONS), steps=0,
             colours=COLOUR, names={c: c for c in CONDITIONS},
             min_snr_ratio=2.0, rest_from=2, with_p1=True, traces=True,
             title=m, save=None, **KW)

### 4d · Across participants — mean ± SD, muscle by muscle

The figures above are one muscle at a time. This pools them: for each muscle, the **mean over the
three participants** with **SD**, and every participant as a **dot in its bar's colour**.

**Top figure — what lidocaine did.** The response with lidocaine as a % of the same muscle's
response before, both read at the same intensity. The pale bar is **before**, 100 % by
construction; grey is 30 Hz with lidocaine, red is ARC-EX with lidocaine. **100 % = the skin block
cost that protocol nothing.**

**Bottom figures — what the train did**, before and with lidocaine separately: the mean of pulses
2–10 as a % of each train's own 1st pulse. Read the pair to see whether blocking cutaneous
afferents changed how much the response falls along the burst — a reflex component removed should
show up here even where the 1st pulse barely moves.

`n` sits on each bar: a reading whose 1st pulse is under `2 x` its own noise is left out and named
underneath, since all of these divide by that pulse.

In [ ]:
# what lidocaine cost each protocol, pooled over participants
fig_group_summary(tab, SUBJECTS, MUSCLES, CONDITIONS, base=BASE, key="pct",
                  colours=COLOUR, min_snr_ratio=2.0, err="sd",
                  reference=("before lidocaine", 100),
                  ylabel="% of before", ref=100, ylim=(0, 200),
                  title="Lidocaine — response as % of the same muscle before, mean ± SD")

# and how the train behaves in each block, before against with lidocaine
for cond_set, ttl in ((["30 Hz · before", "ARC-EX · before"],       "before lidocaine"),
                      (["30 Hz · lidocaine", "ARC-EX · lidocaine"], "with lidocaine")):
    fig_group_summary(tab, SUBJECTS, MUSCLES, cond_set, key="dep_rest",
                      colours=COLOUR, min_snr_ratio=2.0, err="sd",
                      reference=("1st pulse", 100),
                      ylabel="% of the 1st pulse", ref=100, ylim=(0, 175),
                      title=f"Mean of pulses 2–10, {ttl} — mean ± SD")

### 4e · Before against with lidocaine, one panel per protocol

The same comparison as 4d, laid out so the pair can be read in one glance: **30 Hz on the left,
ARC-EX on the right**, sharing a y axis. In each panel, for each muscle, the **pale** bar is
before lidocaine and the **solid** one is with it, mean over the three participants with SD, every
participant a dot in its bar's colour.

**First figure — how much the response falls along the train** (mean of pulses 2–10 as a % of that
train's own 1st pulse). Both bars are on the same footing here: each train is measured against
itself, so before and with lidocaine can be compared directly and the height difference is what
lidocaine did to the depression.

**Second figure — the same for the 2nd pulse alone**, 33 ms after the first.

Nothing is divided by the other block, so a muscle can appear in one bar and not the other; `n`
on each bar says how many participants went in, and a reading under 2x its own noise is left out
and named underneath.

In [ ]:
PANELS_BY_PROTOCOL = {p: [f"{p} · before", f"{p} · lidocaine"] for p in PROTOCOLS}

for key, lab, ttl in (("dep_rest", "% of the 1st pulse",
                       "How much the response falls along the train — before vs with lidocaine"),
                      ("dep2", "% of the 1st pulse",
                       "The 2nd pulse (33 ms) — before vs with lidocaine")):
    fig_group_panels(tab, SUBJECTS, MUSCLES, PANELS_BY_PROTOCOL, key=key,
                     colours=COLOUR, min_snr_ratio=2.0, err="sd",
                     labels={f"{p} · {ph}": ph for p in PROTOCOLS
                             for ph in ("before", "lidocaine")},
                     ylabel=lab, ref=100, ylim=(0, 175), title=ttl,
                     save=None)   # save=f"figures/across_participants/lido_{key}.png"

#### The whole train, first pulse included

The figures above take each train apart — the 1st pulse, the 2nd, the rest. This puts it back
together: the **average of all 10 pulses** with lidocaine, as a **% of the average of all 10
pulses before**, in the same muscle at the same intensity.

It is the closest thing to "how much EMG did the burst deliver in total, and did lidocaine change
it". **100 % = the burst delivered the same overall.** The pale bar is *before*, 100 % by
construction; the solid bar is *with lidocaine*.

Use it alongside 4d rather than instead of it. 4d compares **first pulses**, which is the
recruitment at the start of the train; this compares the **whole train**, which is what the muscle
actually received. A manipulation can leave the first response untouched and still change the
total, or the reverse.

In [ ]:
fig_group_panels(tab, SUBJECTS, MUSCLES, PANELS_BY_PROTOCOL, key="pct_all",
                 colours=COLOUR, min_snr_ratio=2.0, err="sd",
                 labels={f"{p} · {ph}": ph for p in PROTOCOLS
                         for ph in ("before", "lidocaine")},
                 ylabel="% of before (all 10 pulses)", ref=100, ylim=(0, 200),
                 title="Average of all 10 pulses, with lidocaine as % of before",
                 save=None)   # save="figures/across_participants/lido_all10.png"

## 5 · Change a threshold

**The six cells under "these set the analysis" are the ones that change a result.** Each group is
read at its own `before` threshold, so re-picking a `before` moves every number in section 4 for
that participant and protocol.

The six `lidocaine` cells feed section 4a — how far the threshold itself moved — and section 2's
grid. They do not set the intensity any response is measured at.

Move the slider, press **Set to slider**, then **Save**. Re-run section 2 and everything
follows.

In [ ]:
def repick(key):
    """Open the picker for one recording of this comparison, with a Save button.

    The green line is the DETECTOR's answer for this recording - not another block's, and not
    whatever is already saved (that is the orange trace)."""
    csv = runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

### These set the analysis — each group's `before lidocaine` control

In [ ]:
repick(('P04', '30 Hz · before'))

In [ ]:
repick(('P04', 'ARC-EX · before'))

In [ ]:
repick(('P05', '30 Hz · before'))

In [ ]:
repick(('P05', 'ARC-EX · before'))

In [ ]:
repick(('P06', '30 Hz · before'))

In [ ]:
repick(('P06', 'ARC-EX · before'))

### The lidocaine blocks

These feed section 4a, the threshold shift, and section 2's grid. They do not set the intensity any response is read at.

In [ ]:
repick(('P04', '30 Hz · lidocaine'))

In [ ]:
repick(('P04', 'ARC-EX · lidocaine'))

In [ ]:
repick(('P05', '30 Hz · lidocaine'))

In [ ]:
repick(('P05', 'ARC-EX · lidocaine'))

In [ ]:
repick(('P06', '30 Hz · lidocaine'))

In [ ]:
repick(('P06', 'ARC-EX · lidocaine'))